# EXP012 — locked retrospective NASA evaluation V1

**Preparation audited and ready. This notebook is a separate runner; it never trains or recalibrates. Both execution switches default False.** Read the attached final readiness audit. The original preparation files and scientific settings remain unchanged. The NASA data are historically exposed; this is a locked retrospective comparison.

Run in two deliberate stages:

1. **Phase A: A100-SXM4-80GB.** Set `EXECUTE_LABEL_BLIND_TEST=True` and keep `EVALUATE_LOCKED_EXPORTS=False`; Run all. Upload the new runner bundle, the original EXP012 input bundle, completed `EXP012_PREPARATION_results.zip`, and original NASA `archive.zip` when prompted. Matching cached Drive files are reused. No EXP005 sidecar is needed. Do not upload labels yet.
2. Once `SCORE_EXPORT_FREEZE.json` exists, set `EXECUTE_LABEL_BLIND_TEST=False`, `EVALUATE_LOCKED_EXPORTS=True`; Run all. **CPU is sufficient** for this stage, or keep the same A100 session. Only now upload the original `labeled_anomalies.csv`. The report archive downloads at completion.

Keep the same Drive project folder. Completed score units are reused; an existing complete export is only verified. The full original software/GPU identity is mandatory for Phase A; a mismatch stops safely, without changing frozen settings. Stage B requires Python3.13.15 and NumPy2.1.3. Save the actually executed notebook with the score archive and reports. Stale locks may be released only after terminating the old runtime.


In [ ]:
EXECUTE_LABEL_BLIND_TEST = False
EVALUATE_LOCKED_EXPORTS = False
DRIVE_PROJECT = "EXP012_RETROSPECTIVE_EVALUATION_V1"
PREPARATION_DRIVE_PROJECT = "EXP012_ANOMALY_BENCHMARK_PREPARATION_V1"
RELEASE_STALE_EXPORT_LOCK = False
RELEASE_STALE_LABEL_LOCK = False

import os,sys,json,hashlib,zipfile,time,subprocess,shutil,stat,platform
from pathlib import Path
from google.colab import drive,files
ACTIVE = EXECUTE_LABEL_BLIND_TEST or EVALUATE_LOCKED_EXPORTS
if EXECUTE_LABEL_BLIND_TEST and EVALUATE_LOCKED_EXPORTS:
    raise RuntimeError("Run the two stages separately: seal label-blind exports first, then score labels.")
EXPECTED = {'EXP012_RETROSPECTIVE_EVALUATION_V1_runner_bundle.zip': '31211604e069247aff113355af2b2ff6f1392f2bc9146e7dee94a0f409014d65', 'EXP012_ANOMALY_BENCHMARK_PREPARATION_V1_input_bundle.zip': '8e034443158e5862eae83efa695f570294fe4f9a5f89791604e41b1f195e2c84', 'EXP012_PREPARATION_results.zip': 'cf8a0887e5424f1e854a9114297130d5a68e5204991a5169bb7ff628e3e8ae45', 'archive.zip': '6084d3ee3906381f2c98aa3773b6b2d77c82413503faa78f962196582e873733'}

def file_sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1<<20),b""):h.update(block)
    return h.hexdigest()
def extract_safe(path,dst):
    dst=Path(dst);dst.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        if len(z.namelist())!=len(set(z.namelist())):raise RuntimeError("Duplicate archive paths")
        for item in z.infolist():
            p=(dst/item.filename).resolve()
            if dst.resolve() not in p.parents or stat.S_ISLNK(item.external_attr>>16):raise RuntimeError("Unsafe archive path")
            if item.is_dir():p.mkdir(parents=True,exist_ok=True)
            else:p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(item.filename))
def verify_files(root,manifest):
    m=json.loads((root/manifest).read_text())
    actual={p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file() and "__pycache__" not in p.parts and p.name!=manifest}
    if actual!=set(m["files"]):raise RuntimeError("File roster differs")
    for n,h in m["files"].items():
        if file_sha(root/n)!=h:raise RuntimeError("Changed bytes: "+n)
    return m

if ACTIVE:
    drive.mount("/content/drive")
    PROJECT=Path("/content/drive/MyDrive")/DRIVE_PROJECT
    INPUTS=PROJECT/"inputs";LOGS=PROJECT/"logs";SCORES=PROJECT/"scores";REPORTS=PROJECT/"label_reports";EXPORTS=PROJECT/"exports"
    for p in [INPUTS,LOGS,EXPORTS]:p.mkdir(parents=True,exist_ok=True)
    PREP=Path("/content/drive/MyDrive")/PREPARATION_DRIVE_PROJECT
    candidates={n:INPUTS/n for n in EXPECTED}
    # Original preparation Drive files are read-only candidates, not regenerated outputs.
    prior={"EXP012_ANOMALY_BENCHMARK_PREPARATION_V1_input_bundle.zip":PREP/"inputs/EXP012_ANOMALY_BENCHMARK_PREPARATION_V1_input_bundle.zip",
           "EXP012_PREPARATION_results.zip":PREP/"results/exports/EXP012_PREPARATION_results.zip"}
    needed=list(EXPECTED) if EXECUTE_LABEL_BLIND_TEST else ["EXP012_RETROSPECTIVE_EVALUATION_V1_runner_bundle.zip"]
    for name in needed:
        if not candidates[name].exists() and name in prior and prior[name].exists():candidates[name]=prior[name]
    missing=[n for n in needed if not candidates[n].exists()]
    if missing:
        print("Upload these ZIP files only (no labels):",missing)
        uploaded=files.upload()
        for name,blob in uploaded.items():
            if Path(name).suffix.lower()!=".zip" or "labeled_anomal" in name.lower() or "labelled_anomal" in name.lower():
                raise RuntimeError("Only the listed ZIP inputs are accepted before the score seal. Do not upload labels yet.")
            p=Path("/content")/Path(name).name;p.write_bytes(blob);h=file_sha(p)
            matched=[n for n in needed if EXPECTED[n]==h]
            if len(matched)!=1:raise RuntimeError("Unrecognised input bytes: "+name)
            target=INPUTS/matched[0]
            if target.exists():
                if file_sha(target)!=h:raise RuntimeError("Existing input differs; never overwrite it")
            else:
                tmp=target.with_suffix(".part");shutil.copyfile(p,tmp);os.replace(tmp,target)
            candidates[matched[0]]=target
        del uploaded
    for name in needed:
        if not candidates[name].exists():raise FileNotFoundError("Required existing file missing: "+name)
        if file_sha(candidates[name])!=EXPECTED[name]:raise RuntimeError("Input checksum differs: "+name)
    WORK=Path("/content/EXP012_EVALUATION_WORK");WORK.mkdir(exist_ok=True)
    RUNNER=WORK/"runner"
    if RUNNER.exists():shutil.rmtree(RUNNER) # Ephemeral code extraction only; Drive results are preserved.
    extract_safe(candidates["EXP012_RETROSPECTIVE_EVALUATION_V1_runner_bundle.zip"],RUNNER)
    verify_files(RUNNER,"RUNNER_MANIFEST.json")
    if EXECUTE_LABEL_BLIND_TEST:
        PACKAGE=WORK/"original_package";PREP_RESULTS=WORK/"completed_preparation"
        for p,name,manifest in [(PACKAGE,"EXP012_ANOMALY_BENCHMARK_PREPARATION_V1_input_bundle.zip","PACKAGE_MANIFEST.json"),(PREP_RESULTS,"EXP012_PREPARATION_results.zip","OUTPUT_MANIFEST.json")]:
            if p.exists():verify_files(p,manifest)
            else:extract_safe(candidates[name],p);verify_files(p,manifest)
    print("New runner and existing inputs verified. No training or preparation rerun.")
else:
    print("Both stages disabled. Read the audit and set exactly one execution switch when ready.")


## Isolated pinned environment

Phase A provisions the complete original resolved lock and then compares the actual CUDA/environment identity with the completed EXP012 run. Phase B alone provisions only the pinned CPU label-metrics interpreter/NumPy. Bootstrap packages are outside the scientific environment; commands and actual versions are recorded. No frozen scientific source is patched.


In [ ]:
if ACTIVE:
    setup_log=LOGS/("setup_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".log")
    COMMANDS=[]
    def logged(command,env=None,log_path=None):
        log_path=log_path or setup_log;COMMANDS.append(command)
        with open(log_path,"a",buffering=1) as log:
            log.write("COMMAND "+repr(command)+"\n")
            process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
            try:
                for line in process.stdout:print(line,end="");log.write(line)
                status=process.wait()
            except BaseException:
                process.terminate()
                try:process.wait(timeout=30)
                except subprocess.TimeoutExpired:process.kill();process.wait()
                raise
            if status:raise RuntimeError("Safe stop; command failed. Evidence: "+str(log_path))
    GPU="CPU label-metrics stage"
    if EXECUTE_LABEL_BLIND_TEST:
        GPU=subprocess.check_output(["nvidia-smi","--query-gpu=name,driver_version,uuid","--format=csv,noheader"],text=True).strip()
        if "A100-SXM4-80GB" not in GPU:raise RuntimeError("Phase A requires the completed EXP012 A100-SXM4-80GB identity; choose that runtime.")
    BOOT=Path("/content/exp012_evaluation_bootstrap")
    if not (BOOT/"virtualenv").exists():
        logged([sys.executable,"-m","pip","install","--disable-pip-version-check","--target",str(BOOT),"virtualenv","uv"])
    BOOT_ENV=os.environ.copy();BOOT_ENV["PYTHONPATH"]=str(BOOT)
    bootstrap_versions=subprocess.check_output([sys.executable,"-c","import importlib.metadata as m,json;print(json.dumps({d.metadata['Name']:d.version for d in m.distributions(path=['/content/exp012_evaluation_bootstrap'])},sort_keys=True))"],text=True)
    interpreter=sys.executable
    if platform.python_version()!="3.13.15":
        uv=BOOT/"bin/uv";logged([str(uv),"python","install","3.13.15"],env=BOOT_ENV)
        interpreter=subprocess.check_output([str(uv),"python","find","3.13.15"],text=True,env=BOOT_ENV).strip()
    if subprocess.check_output([interpreter,"-c","import platform;print(platform.python_version())"],text=True).strip()!="3.13.15":raise RuntimeError("Pinned interpreter unavailable")
    ENV=WORK/"scientific_venv";PYTHON=ENV/"bin/python";READY=ENV/"EXP012_EVALUATION_READY.json"
    if not PYTHON.exists():logged([sys.executable,"-m","virtualenv","--python",interpreter,str(ENV)],env=BOOT_ENV)
    LOCK=PACKAGE/"provenance/requirements-resolved.txt" if EXECUTE_LABEL_BLIND_TEST else RUNNER/"requirements-labels.txt"
    previous=json.loads(READY.read_text()) if READY.exists() else {}
    if previous.get("requirements_sha256")!=file_sha(LOCK):
        command=[str(PYTHON),"-m","pip","install","--disable-pip-version-check","-r",str(LOCK)]
        if EXECUTE_LABEL_BLIND_TEST:command += ["--extra-index-url","https://download.pytorch.org/whl/cu130"]
        logged(command)
    logged([str(PYTHON),"-m","pip","check"])
    resolved=subprocess.check_output([str(PYTHON),"-m","pip","freeze","--all"],text=True)
    if EXECUTE_LABEL_BLIND_TEST and sorted(resolved.splitlines())!=sorted(LOCK.read_text().splitlines()):
        (LOGS/"ACTUAL_REQUIREMENTS_MISMATCH.txt").write_text(resolved)
        raise RuntimeError("Scientific dependency lock differs; no TEST inference started")
    metrics_env=json.loads(subprocess.check_output([str(PYTHON),"-c","import numpy,platform,json;print(json.dumps({'python':platform.python_version(),'numpy':numpy.__version__}))"],text=True))
    if metrics_env!={"python":"3.13.15","numpy":"2.1.3"}:raise RuntimeError("Pinned metrics environment differs")
    READY.write_text(json.dumps({"requirements_sha256":file_sha(LOCK)}))
    RUN_ENV=os.environ.copy();RUN_ENV.pop("PYTHONPATH",None)
    RUN_ENV.update(PYTHONHASHSEED="17",CUBLAS_WORKSPACE_CONFIG=":4096:8",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",OPENBLAS_NUM_THREADS="1",PYTHONUNBUFFERED="1",PYTHONDONTWRITEBYTECODE="1")
    provision={"version":"EXP012_RETROSPECTIVE_EVALUATION_V1","stage":"A_SCORE_EXPORT" if EXECUTE_LABEL_BLIND_TEST else "B_LABEL_METRICS",
               "host_python":platform.python_version(),"interpreter":interpreter,"interpreter_sha256":file_sha(interpreter),"bootstrap_packages":json.loads(bootstrap_versions),
               "commands":COMMANDS,"requirements_sha256":file_sha(LOCK),"actual_requirements":resolved,"GPU_session":GPU,"runner_archive_sha256":EXPECTED["EXP012_RETROSPECTIVE_EVALUATION_V1_runner_bundle.zip"],"setup_log_sha256":file_sha(setup_log)}
    try:
        from google.colab import _message
        snap=LOGS/("notebook_at_launch_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".json")
        snap.write_text(json.dumps(_message.blocking_request("get_ipynb",timeout_sec=30)));provision["notebook_snapshot_sha256"]=file_sha(snap)
    except Exception as e:provision["notebook_snapshot_status"]="Save executed notebook manually: "+type(e).__name__
    (LOGS/("provisioning_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".json")).write_text(json.dumps(provision,indent=2,sort_keys=True))
    def release_lock(path,requested):
        if path.exists() and requested:
            old=json.loads(path.read_text());receipt=LOGS/("released_"+path.parent.name+"_lock_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".json")
            receipt.write_text(json.dumps({"old_runtime_terminated_before_release":True,"prior_lock":old},indent=2));path.unlink()
        if path.exists():raise RuntimeError("Live/stale lock exists. Terminate the old runtime before explicitly releasing its lock.")
    print("Pinned environment provisioned. Scientific settings unchanged.")


## Phase A — label-blind TEST score export

The explicit switch performs TEST inference with existing weights. Both frozen detectors are applied, forecasts repeated exactly, state hashes checked, and every score/flag/actual issue time sealed. No labels are requested or opened. The full export stays on Drive and can be scored later on CPU. Raw TEST cache is excluded from the portable score archive.


In [ ]:
if EXECUTE_LABEL_BLIND_TEST:
    release_lock(SCORES/"ACTIVE_RUN.json",RELEASE_STALE_EXPORT_LOCK);RELEASE_STALE_EXPORT_LOCK=False
    log=LOGS/("phase_A_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".log")
    logged([str(PYTHON),str(RUNNER/"src/export_scores.py"),"--results",str(PREP_RESULTS),"--package",str(PACKAGE),"--nasa-archive",str(candidates["archive.zip"]),"--output",str(SCORES),"--execute-test"],env=RUN_ENV,log_path=log)
    seal=json.loads((SCORES/"SCORE_EXPORT_FREEZE.json").read_text())
    if seal["status"]!="COMPLETE_LABEL_BLIND_EXPORT_LOCKED" or seal["NASA_labels_opened"] or seal["unique_streams"]!=1053:raise RuntimeError("Incomplete score seal")
    score_zip=EXPORTS/"EXP012_LABEL_BLIND_SCORE_EXPORT.zip";receipt=EXPORTS/(score_zip.name+".sha256")
    if score_zip.exists():
        if not receipt.exists() or file_sha(score_zip)!=receipt.read_text().split()[0]:raise RuntimeError("Existing score archive checksum differs")
    else:
        tmp=score_zip.with_suffix(".part")
        with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED) as z:
            names=list(seal["files"])+["SCORE_EXPORT_FREEZE.json","EVALUATION_RUN_FREEZE.json","TEST_INPUT_FREEZE.json"]
            if (SCORES/"LABEL_BLIND_ACCESS_EVENTS.jsonl").exists():names += ["LABEL_BLIND_ACCESS_EVENTS.jsonl"]
            for name in names:z.write(SCORES/name,name)
        os.replace(tmp,score_zip);receipt.write_text(file_sha(score_zip)+"  "+score_zip.name+"\n")
    print("LABEL-BLIND EXPORT COMPLETE; labels have not been opened.")
    print("Score seal SHA256:",file_sha(SCORES/"SCORE_EXPORT_FREEZE.json"))
    print("Portable score export:",score_zip,"SHA256",receipt.read_text().split()[0])
    print("Next: set EXECUTE_LABEL_BLIND_TEST=False and EVALUATE_LOCKED_EXPORTS=True. CPU is sufficient for Phase B.")
else:
    print("Phase A disabled.")


## Phase B — evaluate immutable exports, then open historical labels

This stage checks the complete score seal before requesting the CSV. It never imports models, opens NASA arrays, retrains, recalibrates or replaces detectors. The original label SHA256 and first-P-2 policy are enforced. Event overlap, actual notification timing, strict point metrics and tied-score AP stay separate. Reports include original events outside support, undefined metrics and all prespecified channel-paired bootstrap contrasts.


In [ ]:
if EVALUATE_LOCKED_EXPORTS:
    if not (SCORES/"SCORE_EXPORT_FREEZE.json").exists():
        # Supports a CPU-only session with a portable previously sealed export.
        archive=EXPORTS/"EXP012_LABEL_BLIND_SCORE_EXPORT.zip"
        if not archive.exists():
            print("Upload the previously sealed EXP012_LABEL_BLIND_SCORE_EXPORT.zip only; no labels yet.")
            uploaded=files.upload()
            if len(uploaded)!=1:raise RuntimeError("One sealed score archive required")
            name,blob=next(iter(uploaded.items()))
            if Path(name).suffix.lower()!=".zip":raise RuntimeError("Expected sealed score ZIP")
            p=Path("/content")/Path(name).name;p.write_bytes(blob)
            with zipfile.ZipFile(p) as z:
                if "SCORE_EXPORT_FREEZE.json" not in z.namelist():raise RuntimeError("Not a sealed score export")
            if not archive.exists():shutil.copyfile(p,archive)
            del uploaded
        if SCORES.exists() and any(SCORES.iterdir()):raise RuntimeError("Incomplete score directory exists; resume Phase A instead of replacing it")
        extract_safe(archive,SCORES)
    # Read/verify every exported unit BEFORE asking for labels, with the pinned CPU interpreter.
    validation="import sys,json;from pathlib import Path;sys.path.insert(0,sys.argv[1]);from common import sha,verify_manifest;from export_scores import verify_scores;r=Path(sys.argv[1]).parent;verify_manifest(r,'RUNNER_MANIFEST.json');s=verify_scores(sys.argv[2]);b=json.loads((r/'EVALUATION_BINDING.json').read_text());assert s['preparation_binding_sha256']==sha(r/'EVALUATION_BINDING.json') and s['runner_manifest_sha256']==sha(r/'RUNNER_MANIFEST.json') and s['contract']==b['evaluation_contract'];print('All sealed exports verified before labels')"
    logged([str(PYTHON),"-c",validation,str(RUNNER/"src"),str(SCORES)],env=RUN_ENV)
    release_lock(REPORTS/"ACTIVE_RUN.json",RELEASE_STALE_LABEL_LOCK);RELEASE_STALE_LABEL_LOCK=False
    LABELS=INPUTS/"labeled_anomalies.csv"
    if not (REPORTS/"LABEL_EVALUATION_COMPLETE.json").exists() and not LABELS.exists():
        print("The score seal passed. Now upload the ORIGINAL labeled_anomalies.csv only.")
        uploaded=files.upload()
        if len(uploaded)!=1:raise RuntimeError("Exactly one original label CSV required")
        name,blob=next(iter(uploaded.items()));p=Path("/content")/Path(name).name;p.write_bytes(blob)
        expected=json.loads((RUNNER/"EVALUATION_BINDING.json").read_text())["evaluation_contract"]["historical_labels_sha256"]
        if file_sha(p)!=expected:raise RuntimeError("Label CSV checksum differs from frozen contract")
        tmp=LABELS.with_suffix(".part");shutil.copyfile(p,tmp);os.replace(tmp,LABELS);del uploaded
    log=LOGS/("phase_B_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".log")
    logged([str(PYTHON),str(RUNNER/"src/evaluate_labels.py"),"--scores",str(SCORES),"--labels",str(LABELS),"--output",str(REPORTS),"--evaluate-locked-exports"],env=RUN_ENV,log_path=log)
    complete=verify_files(REPORTS,"LABEL_EVALUATION_COMPLETE.json")
    if complete["status"]!="COMPLETE_LOCKED_RETROSPECTIVE_EVALUATION":raise RuntimeError("Incomplete label evaluation")
    report_zip=EXPORTS/"EXP012_RETROSPECTIVE_EVALUATION_REPORTS.zip";receipt=EXPORTS/(report_zip.name+".sha256")
    if report_zip.exists():
        if not receipt.exists() or file_sha(report_zip)!=receipt.read_text().split()[0]:raise RuntimeError("Existing report archive checksum differs")
    else:
        tmp=report_zip.with_suffix(".part")
        with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED) as z:
            for name in [*complete["files"],"LABEL_EVALUATION_COMPLETE.json"]:z.write(REPORTS/name,name)
        os.replace(tmp,report_zip);receipt.write_text(file_sha(report_zip)+"  "+report_zip.name+"\n")
    print("Locked retrospective evaluation complete:",REPORTS)
    print("Do not retune or select a detector/model in response to these results.")
    files.download(str(report_zip))
else:
    print("Phase B disabled. No label CSV requested or opened.")


## Preserve execution evidence

Save the actually executed notebook (File → Download → .ipynb). The notebook also captures best-effort launch/final snapshots and setup/execution logs. These remain separate from frozen scientific outputs. Keep the labelled reports together with the score seal, original benchmark contract and audit.


In [ ]:
if ACTIVE:
    try:
        from google.colab import _message
        snapshot=_message.blocking_request("get_ipynb",timeout_sec=30);executed=snapshot.get("ipynb",snapshot)
        if "cells" not in executed:raise ValueError("Notebook payload unavailable")
        (LOGS/("ACTUALLY_EXECUTED_"+time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+".ipynb")).write_text(json.dumps(executed,indent=1))
    except Exception as e:print("Save the actually executed notebook manually:",type(e).__name__)
    evidence=PROJECT/"EXP012_RETROSPECTIVE_execution_provenance.zip"
    with zipfile.ZipFile(evidence.with_suffix(".part"),"w",zipfile.ZIP_DEFLATED) as z:
        for p in sorted(LOGS.rglob("*")):
            if p.is_file():z.write(p,p.relative_to(LOGS))
    os.replace(evidence.with_suffix(".part"),evidence)
    (PROJECT/(evidence.name+".sha256")).write_text(file_sha(evidence)+"  "+evidence.name+"\n")
    print("Drive project:",PROJECT)
    print("Preserve score export, report archive, execution evidence and actual notebook.")
